# Register File Area Scaling — IHP SG13G2

Reproducing (and extending) the results from:
> *"Exploring the Design Space of Register Files"* (ACM 1998, DOI 10.1145/280756.280943)

using **Yosys** synthesis targeting the **IHP SG13G2 130 nm** standard-cell library.

## Setup
Run the sweep first:
```bash
python flow/run_sweep.py --sweep all
```
then open this notebook.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

# Resolve project root robustly whether notebook runs from repo root or notebooks/
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS = RESULTS_DIR / 'summary.csv'

assert RESULTS.exists(), f'Run the sweep first: python flow/run_sweep.py'

df = pd.read_csv(RESULTS)
df = df[df['status'] == 'OK'].copy()
print(f'Loaded {len(df)} successful runs from {RESULTS}.')
df.head()

---
## 1 · Port Sweep — Area vs. Read / Write Port Count

The paper predicts area ∝ N · (Nr + Nw)² for a custom-VLSI register file.  
Below we check whether this quadratic scaling survives standard-cell synthesis.

In [ ]:
port = df[df['sweep'] == 'port'].copy()
port['total_ports'] = port['NUM_RD_PORTS'] + port['NUM_WR_PORTS']

# --- Matplotlib ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for nw, grp in port.groupby('NUM_WR_PORTS'):
    grp_s = grp.sort_values('NUM_RD_PORTS')
    axes[0].plot(grp_s['NUM_RD_PORTS'], grp_s['chip_area_um2'],
                 marker='o', label=f'Nw={nw}')
axes[0].set_xlabel('Read ports (Nr)')
axes[0].set_ylabel('Chip area (µm²)')
axes[0].set_title('Area vs Read ports')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for nr, grp in port.groupby('NUM_RD_PORTS'):
    grp_s = grp.sort_values('NUM_WR_PORTS')
    axes[1].plot(grp_s['NUM_WR_PORTS'], grp_s['chip_area_um2'],
                 marker='s', label=f'Nr={nr}')
axes[1].set_xlabel('Write ports (Nw)')
axes[1].set_ylabel('Chip area (µm²)')
axes[1].set_title('Area vs Write ports')
axes[1].legend(ncol=2, fontsize=9)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Port Sweep — IHP SG13G2, 32 regs × 32-bit', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'port_sweep.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Plotly (interactive) ---
fig_px = px.line(
    port.sort_values(['NUM_WR_PORTS', 'NUM_RD_PORTS']),
    x='NUM_RD_PORTS', y='chip_area_um2',
    color='NUM_WR_PORTS', symbol='NUM_WR_PORTS',
    markers=True,
    hover_data=['NUM_RD_PORTS', 'NUM_WR_PORTS', 'chip_area_um2', 'ff_count'],
    labels={
        'NUM_RD_PORTS': 'Read ports (Nr)',
        'chip_area_um2': 'Chip area (µm²)',
        'NUM_WR_PORTS': 'Write ports',
    },
    title='Area vs Read ports — IHP SG13G2, 32×32-bit regfile',
)
fig_px.update_layout(template='plotly_white', hovermode='x unified')
fig_px.show()

### 1a · Log-log fit: verify quadratic scaling in total port count

In [ ]:
port_fit = port[port['chip_area_um2'].notna()].copy()
port_fit['log_P'] = np.log(port_fit['total_ports'])
port_fit['log_A'] = np.log(port_fit['chip_area_um2'])

coef = np.polyfit(port_fit['log_P'], port_fit['log_A'], 1)
print(f'Log-log slope (area ~ P^α):  α = {coef[0]:.3f}  (paper predicts ≈2.0)')

# Overlay fitted curve
P_range = np.linspace(port_fit['total_ports'].min(), port_fit['total_ports'].max(), 100)
A_fit   = np.exp(coef[1]) * P_range ** coef[0]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(port_fit['total_ports'], port_fit['chip_area_um2'],
           s=60, zorder=5, label='Synthesis data')
ax.plot(P_range, A_fit, 'r--', label=f'Power-law fit  α={coef[0]:.2f}')
ax.set_xlabel('Total ports  P = Nr + Nw')
ax.set_ylabel('Chip area (µm²)')
ax.set_title('Area scaling with total port count')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'port_scaling_fit.png', bbox_inches='tight')
plt.show()

---
## 2 · Bit-width Sweep — Linear Scaling Check

In [ ]:
bw = df[df['sweep'] == 'bitwidth'].sort_values('BITWIDTH').copy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(bw['BITWIDTH'], bw['chip_area_um2'], marker='o', linewidth=2)
# Linear reference
bw_ref = bw.iloc[0]
ax.plot(bw['BITWIDTH'], bw_ref['chip_area_um2'] / bw_ref['BITWIDTH'] * bw['BITWIDTH'],
        'k--', alpha=0.5, label='Linear reference')
ax.set_xlabel('Bit-width')
ax.set_ylabel('Chip area (µm²)')
ax.set_title('Area vs Bit-width (2R1W, 32 regs)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'bitwidth_sweep.png', bbox_inches='tight')
plt.show()

fig_bw = px.scatter(
    bw, x='BITWIDTH', y='chip_area_um2',
    hover_data=['BITWIDTH', 'chip_area_um2', 'ff_count'],
    trendline='ols',
    title='Area vs Bit-width — IHP SG13G2',
    labels={'BITWIDTH': 'Bit-width', 'chip_area_um2': 'Chip area (µm²)'},
)
fig_bw.update_layout(template='plotly_white')
fig_bw.show()

---
## 3 · Register Count Sweep

In [ ]:
rc = df[df['sweep'] == 'regcount'].sort_values('NUM_REGS').copy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rc['NUM_REGS'], rc['chip_area_um2'], marker='^', linewidth=2, label='Synthesis')
rc_ref = rc.iloc[0]
ax.plot(rc['NUM_REGS'], rc_ref['chip_area_um2'] / rc_ref['NUM_REGS'] * rc['NUM_REGS'],
        'k--', alpha=0.5, label='Linear reference')
ax.set_xlabel('Number of registers')
ax.set_ylabel('Chip area (µm²)')
ax.set_title('Area vs Register count (2R1W, 32-bit)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'regcount_sweep.png', bbox_inches='tight')
plt.show()

fig_rc = px.line(
    rc, x='NUM_REGS', y='chip_area_um2',
    markers=True,
    hover_data=['NUM_REGS', 'chip_area_um2', 'ff_count'],
    title='Area vs Register count — IHP SG13G2',
    labels={'NUM_REGS': 'Register count', 'chip_area_um2': 'Chip area (µm²)'},
)
fig_rc.update_layout(template='plotly_white')
fig_rc.show()

---
## 4 · Banking Sweep — Effect of Read-Port Banking

Banking replicates the register storage but reduces the mux fanout per bank.  
We expect an area crossover: at high read-port counts, banking should win.

In [ ]:
bank = df[df['sweep'] == 'banking'].copy()

# --- Matplotlib ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for nr, grp in bank.groupby('NUM_RD_PORTS'):
    for nw, g2 in grp.groupby('NUM_WR_PORTS'):
        g2s = g2.sort_values('NUM_BANKS')
        axes[0].plot(g2s['NUM_BANKS'], g2s['chip_area_um2'],
                     marker='o', label=f'Nr={nr}, Nw={nw}')
axes[0].set_xlabel('Number of banks')
axes[0].set_ylabel('Chip area (µm²)')
axes[0].set_title('Area vs Banking factor')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Normalised area (relative to 1-bank baseline for the same Nr/Nw)
def normalise(grp):
    base = grp.loc[grp['NUM_BANKS'] == grp['NUM_BANKS'].min(), 'chip_area_um2'].values
    if len(base) == 0 or base[0] is None:
        return grp
    grp = grp.copy()
    grp['area_norm'] = grp['chip_area_um2'] / base[0]
    return grp

bank_norm = bank.groupby(['NUM_RD_PORTS', 'NUM_WR_PORTS'], group_keys=False).apply(normalise)
for nr, grp in bank_norm.groupby('NUM_RD_PORTS'):
    for nw, g2 in grp.groupby('NUM_WR_PORTS'):
        g2s = g2.sort_values('NUM_BANKS')
        axes[1].plot(g2s['NUM_BANKS'], g2s['area_norm'],
                     marker='s', label=f'Nr={nr}, Nw={nw}')
axes[1].axhline(1.0, color='k', linestyle='--', alpha=0.4, label='No banking baseline')
axes[1].set_xlabel('Number of banks')
axes[1].set_ylabel('Normalised area')
axes[1].set_title('Normalised area vs Banking factor')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

fig.suptitle('Banking Sweep — IHP SG13G2, 32 regs × 32-bit', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'banking_sweep.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Plotly (interactive banking) ---
bank['config'] = bank.apply(
    lambda r: f"Nr={int(r.NUM_RD_PORTS)}, Nw={int(r.NUM_WR_PORTS)}", axis=1
)

fig_bank = px.line(
    bank.sort_values('NUM_BANKS'),
    x='NUM_BANKS', y='chip_area_um2',
    color='config', symbol='config',
    markers=True,
    hover_data=['NUM_BANKS', 'NUM_RD_PORTS', 'NUM_WR_PORTS',
                'chip_area_um2', 'ff_count'],
    title='Area vs Banking factor — IHP SG13G2, 32×32-bit regfile',
    labels={
        'NUM_BANKS':     'Number of banks',
        'chip_area_um2': 'Chip area (µm²)',
        'config':        'Port config',
    },
)
fig_bank.update_layout(template='plotly_white', hovermode='x unified')
fig_bank.show()

---
## 5 · Summary Heatmap — Area across all (Nr, Nw) configurations

In [ ]:
pivot = port.pivot_table(
    values='chip_area_um2', index='NUM_WR_PORTS', columns='NUM_RD_PORTS'
)

# Matplotlib heatmap
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('Read ports (Nr)')
ax.set_ylabel('Write ports (Nw)')
ax.set_title('Chip area (µm²) — 32 regs × 32-bit')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                    fontsize=8, color='white')
plt.colorbar(im, ax=ax, label='µm²')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'area_heatmap.png', bbox_inches='tight')
plt.show()

# Plotly heatmap
fig_hm = px.imshow(
    pivot,
    text_auto='.0f',
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Chip area (µm²) heatmap — 32×32-bit regfile',
    labels={
        'x': 'Read ports (Nr)',
        'y': 'Write ports (Nw)',
        'color': 'Area (µm²)',
    },
)
fig_hm.update_layout(template='plotly_white')
fig_hm.show()

---
## 6 · Key Findings

| Question | Paper prediction | Observed? |
|---|---|---|
| Area ∝ P^α, α≈2 | Yes (custom VLSI) | See §1a |
| Write ports more expensive than read ports | Yes | See §1 |
| Area ∝ BITWIDTH (linear) | Yes | See §2 |
| Banking reduces area at high port counts | Yes | See §4 |

*Fill in the observed column after running the sweep.*